# M4d-4C — Verificação manual dos resultados (4 cabeças) no conjunto de teste holdout

Variante de `notebooks/bert/11.verificar_resultados_manualmente.ipynb` para o modelo com encoder compartilhado + 4 cabeças de classificação. Carrega o melhor cenário treinado em `notebooks/bert_4_cabecas/06...ipynb` e roda a inferência sobre os documentos extraídos pelo notebook 08 (em `notebooks/bert/`).

**Inspeção puramente visual/qualitativa**: mostra o que o modelo previu (texto, tipo, página) e desenha as caixas sobre o PDF, sem carregar gabarito. Para números de precision/recall/F1, ver `notebooks/bert_4_cabecas/10.extrair_metricas_holdout_4_cabecas.ipynb`.

**Pré-requisito:** `notebooks/bert/08.extrair_texto_holdout.ipynb` já rodado (kernel `locscan-ocr`). Este notebook roda no kernel GERAL do projeto (`.venv`, torch/transformers).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

print(f"CUDA disponível: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Configuração

`INCLUIR_SINTETICOS` escolhe qual experimento usar (deve bater com `notebooks/bert_4_cabecas/06`). `CENARIO_MANUAL` força um cenário específico em vez do melhor por F1.

In [ ]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()
sys.path.insert(0, str(RAIZ / "scripts"))
from pos_processamento_ner import extrair_nome_assinatura_digital, propagar_entidades_por_texto_identico

INCLUIR_SINTETICOS = False
CENARIO_MANUAL = None  # ex: "c08_warmup" -- None usa o melhor por F1 do experimento
APLICAR_REGEX_ASSINATURA_DIGITAL = True
APLICAR_PROPAGACAO_TEXTO_IDENTICO = True

MAX_LENGTH = 512

PASTA_EXTRACAO_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "extracao"
PASTA_ARQUIVOS_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "arquivos"

_sufixo_experimento = "-com-sinteticos" if INCLUIR_SINTETICOS else "-sem-sinteticos"
PASTA_CENARIOS = RAIZ / "models" / f"bertimbau-4cabecas-pii-v1-cenarios{_sufixo_experimento}"

TIPOS = ["CPF", "RG", "NOME", "ENDERECO"]
LABEL2ID_POR_TIPO = {tipo: {"O": 0, f"B-{tipo}": 1, f"I-{tipo}": 2} for tipo in TIPOS}
ID2LABEL_POR_TIPO = {tipo: {v: k for k, v in LABEL2ID_POR_TIPO[tipo].items()} for tipo in TIPOS}

resultados_cenarios = json.loads((PASTA_CENARIOS / "_comparacao_cenarios.json").read_text(encoding="utf-8"))
cenario_escolhido = next((r for r in resultados_cenarios if r["nome"] == CENARIO_MANUAL), resultados_cenarios[0]) if CENARIO_MANUAL else resultados_cenarios[0]

print(f"Experimento: {PASTA_CENARIOS.name}")
print(f"Cenário: {cenario_escolhido['nome']} (F1 token-level={cenario_escolhido['eval_f1']:.4f})")
print(f"Pasta do modelo: {cenario_escolhido['pasta_modelo']}")

arquivos_extraidos = sorted(PASTA_EXTRACAO_HOLDOUT.glob("*.json"))
print(f"\nDocumentos extraídos no holdout: {len(arquivos_extraidos)}")
if not arquivos_extraidos:
    print(f"[AVISO] Nenhum JSON em {PASTA_EXTRACAO_HOLDOUT} -- rode notebooks/bert/08 primeiro.")

## Carrega o modelo (encoder + 4 cabeças)

In [ ]:
class BERTimbau4Cabecas(nn.Module):
    """Mesma definição de notebooks/bert_4_cabecas/06 -- reproduzida aqui por autocontenção."""

    def __init__(self, checkpoint_base: str, tipos: list):
        super().__init__()
        self.tipos = list(tipos)
        self.encoder = AutoModel.from_pretrained(checkpoint_base)
        hidden = self.encoder.config.hidden_size
        p_dropout = getattr(self.encoder.config, "hidden_dropout_prob", 0.1)
        self.dropout = nn.Dropout(p_dropout)
        self.cabecas = nn.ModuleDict({tipo: nn.Linear(hidden, 3) for tipo in self.tipos})

    def forward(self, input_ids, attention_mask):
        saida_encoder = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        oculto = self.dropout(saida_encoder.last_hidden_state)
        return {"logits_por_tipo": {tipo: self.cabecas[tipo](oculto) for tipo in self.tipos}}

    @classmethod
    def load(cls, pasta, device="cpu"):
        pasta = Path(pasta)
        config = json.loads((pasta / "config_4cabecas.json").read_text(encoding="utf-8"))
        modelo = cls(str(pasta / "encoder"), config["tipos"])
        cabecas_state = torch.load(pasta / "cabecas.pt", map_location=device)
        for tipo in config["tipos"]:
            modelo.cabecas[tipo].load_state_dict(cabecas_state[tipo])
        return modelo.to(device)


tokenizer = AutoTokenizer.from_pretrained(Path(cenario_escolhido["pasta_modelo"]) / "encoder")
model = BERTimbau4Cabecas.load(cenario_escolhido["pasta_modelo"], device=device)
model.eval()
print(f"Modelo carregado: {cenario_escolhido['nome']}")


def agrupar_bboxes_por_linha(tokens_bbox: list, indices: list) -> list:
    if not indices:
        return []
    grupos = []
    grupo_atual = [tokens_bbox[indices[0]]]
    y_ref = tokens_bbox[indices[0]][1]
    for i in indices[1:]:
        bbox = tokens_bbox[i]
        altura_ref = grupo_atual[-1][3] - grupo_atual[-1][1]
        if abs(bbox[1] - y_ref) > max(altura_ref * 0.6, 3):
            grupos.append(grupo_atual)
            grupo_atual = [bbox]
            y_ref = bbox[1]
        else:
            grupo_atual.append(bbox)
    grupos.append(grupo_atual)
    return [[min(b[0] for b in g), min(b[1] for b in g), max(b[2] for b in g), max(b[3] for b in g)] for g in grupos]


from pos_processamento_ner import mesclar_previsoes_por_janela


@torch.no_grad()
def _prever_tags_janela_tipo(tokens_janela: list, tipo: str) -> list:
    encoding = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoding = {k: v.to(device) for k, v in encoding.items()}
    saida = model(**encoding)
    pred_ids = saida["logits_por_tipo"][tipo][0].argmax(dim=-1).tolist()
    word_ids = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
    tags_por_palavra = ["O"] * len(tokens_janela)
    visto = set()
    for pid, word_idx in zip(pred_ids, word_ids):
        if word_idx is None or word_idx in visto:
            continue
        visto.add(word_idx)
        tags_por_palavra[word_idx] = ID2LABEL_POR_TIPO[tipo][pid]
    return tags_por_palavra


def prever_pagina_4cabecas(tokens: list, tokens_bbox: list) -> list:
    """Prevê a página inteira, uma cabeça (tipo) por vez, dividindo em janelas deslizantes
    com sobreposição quando ultrapassa MAX_LENGTH subtokens (achado 2026-09-15, caso MINAS
    CAL/segundo endereço truncado -- ver scripts/pos_processamento_ner.py). Cada cabeça tem
    sua própria sequência de tags, então a mescla por janela roda uma vez por tipo."""
    entidades = []
    for tipo in TIPOS:
        tags_por_palavra = mesclar_previsoes_por_janela(
            tokens, tokenizer, lambda tj, t=tipo: _prever_tags_janela_tipo(tj, t), max_length=MAX_LENGTH
        )
        i, n = 0, len(tags_por_palavra)
        while i < n:
            if tags_por_palavra[i] == "O":
                i += 1
                continue
            indices = [i]
            j = i + 1
            while j < n and tags_por_palavra[j] == f"I-{tipo}":
                indices.append(j)
                j += 1
            entidades.append({
                "type": tipo,
                "bbox": agrupar_bboxes_por_linha(tokens_bbox, indices),
                "indices": indices,
                "text": " ".join(tokens[k] for k in indices),
            })
            i = j
    return entidades


def bbox_externo(lista_bboxes: list):
    if not lista_bboxes:
        return None
    return [min(b[0] for b in lista_bboxes), min(b[1] for b in lista_bboxes),
            max(b[2] for b in lista_bboxes), max(b[3] for b in lista_bboxes)]


def iou(a, b) -> float:
    if a is None or b is None:
        return 0.0
    xi0, yi0 = max(a[0], b[0]), max(a[1], b[1])
    xi1, yi1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi1 - xi0) * max(0, yi1 - yi0)
    if inter == 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    uniao = area_a + area_b - inter
    return inter / uniao if uniao > 0 else 0.0

## Roda a inferência em todos os documentos do holdout e lista as entidades encontradas

In [ ]:
resultados_por_documento = {}

for arq in arquivos_extraidos:
    extracao = json.loads(arq.read_text(encoding="utf-8"))
    doc_id = extracao["document_id"]
    entidades_documento = []

    for pagina_dados in extracao["pages"]:
        tokens_dict = pagina_dados["tokens"]
        if not tokens_dict:
            continue
        tokens = [t["text"] for t in tokens_dict]
        tokens_bbox = [t["bbox"] for t in tokens_dict]

        previstas = prever_pagina_4cabecas(tokens, tokens_bbox)

        if APLICAR_REGEX_ASSINATURA_DIGITAL:
            extras = extrair_nome_assinatura_digital(tokens, tokens_bbox)
            for extra in extras:
                bbox_extra = bbox_externo(extra["bbox"])
                ja_coberto = any(
                    p["type"] == "NOME" and iou(bbox_extra, bbox_externo(p["bbox"])) > 0.1
                    for p in previstas
                )
                if not ja_coberto:
                    previstas.append(extra)

        for p in previstas:
            p["page"] = pagina_dados["page"]
        entidades_documento.extend(previstas)

    if APLICAR_PROPAGACAO_TEXTO_IDENTICO:
        entidades_documento = propagar_entidades_por_texto_identico(
            entidades_documento, extracao["pages"], tipos_propagaveis=tuple(TIPOS)
        )

    resultados_por_documento[doc_id] = entidades_documento

    print(f"\n{'=' * 90}\n{doc_id}\n{'=' * 90}")
    if not entidades_documento:
        print("  (nenhuma entidade encontrada)")
    for e in entidades_documento:
        origem = f" [{e['origem']}]" if "origem" in e else ""
        print(f"  p{e['page']:<3} {e['type']:<9} \"{e['text']}\"{origem}")

n_total = sum(len(v) for v in resultados_por_documento.values())
print(f"\n\nTotal: {len(resultados_por_documento)} documento(s), {n_total} entidade(s) encontrada(s).")

## Visualização: todas as páginas do PDF com as caixas previstas

Preencha `DOC_ID_VISUALIZAR` (use um `document_id` impresso acima) para ver as caixas desenhadas sobre **todas** as páginas do documento, uma figura por página.

In [ ]:
import fitz  # PyMuPDF
from io import BytesIO
from PIL import Image

DOC_ID_VISUALIZAR = next(iter(resultados_por_documento), None)  # troque para outro document_id da lista acima


def localizar_pdf_original(document_id: str) -> Path:
    candidato = PASTA_ARQUIVOS_HOLDOUT / document_id
    if candidato.exists():
        return candidato
    achados = list(PASTA_ARQUIVOS_HOLDOUT.glob(f"{Path(document_id).stem}*"))
    if achados:
        return achados[0]
    raise FileNotFoundError(f"PDF original não encontrado para {document_id} em {PASTA_ARQUIVOS_HOLDOUT}")


CORES_POR_TIPO = {
    "CPF": (0.85, 0.1, 0.1),
    "RG": (0.1, 0.4, 0.85),
    "NOME": (0.1, 0.7, 0.2),
    "ENDERECO": (0.8, 0.6, 0.0),
}

if DOC_ID_VISUALIZAR:
    caminho_pdf = localizar_pdf_original(DOC_ID_VISUALIZAR)
    doc = fitz.open(caminho_pdf)
    entidades_documento = resultados_por_documento[DOC_ID_VISUALIZAR]
    legenda = "  ".join(f"{t}={c}" for t, c in CORES_POR_TIPO.items())

    for pagina_pdf in doc:
        numero_pagina = pagina_pdf.number + 1
        entidades_pagina = [e for e in entidades_documento if e["page"] == numero_pagina]
        for e in entidades_pagina:
            # Um retângulo por linha de e["bbox"] -- ver nota em notebooks/bert/10 (achado
            # 2026-09-15): bbox_externo() sozinho "vaza" para a esquerda em entidades que
            # quebram linha.
            for retangulo in (e.get("bbox") or []):
                pagina_pdf.draw_rect(fitz.Rect(*retangulo), color=CORES_POR_TIPO.get(e["type"], (0, 0, 0)), width=1.5)

        pix = pagina_pdf.get_pixmap(dpi=150)
        img_bytes = pix.tobytes("png")

        plt.figure(figsize=(11, 15))
        plt.imshow(Image.open(BytesIO(img_bytes)))
        plt.axis("off")
        plt.title(f"{DOC_ID_VISUALIZAR} — página {numero_pagina}/{doc.page_count} ({len(entidades_pagina)} entidade(s))\n{legenda}")
        plt.tight_layout()
        plt.show()

    doc.close()
else:
    print("Nenhum documento no holdout -- rode notebooks/bert/08 primeiro.")